# 5 · Build the server disk and installer ISO

## Goal

Construct a bootable server RAW disk, compress it, generate payload digests, build an installer UKI, and package the UEFI ISO. The tested builder runs these operations in one **atomic build transaction** to avoid mismatching server payloads and installer checksums.

## Prerequisites

The committed source tree must be clean, the public FreeIPA CA verified out-of-band, the Docker builder available, and `build/output` absent or empty. The build uses privileged Docker.

## Why the build stays atomic

The container entrypoint (`scripts/build-inside.sh`) now exposes **nine named functions**, each emitting an `SCOS_BUILD_STAGE` marker: verify toolchain → prepare workspace → validate/stage inputs → configure EFI signing → build server RAW → zstd compression → record payload digests → build installer UKI → assemble ISO → record evidence. These are individual, reviewable source blocks but execute as one transaction: the installer must always describe exactly the RAW built in the same run. The checks below inspect each output independently.

In [ ]:
from pathlib import Path
import subprocess, sys
START=Path.cwd().resolve()
ROOT=next((p for p in (START,*START.parents) if (p/"myst.yml").is_file() and (p/"scripts/build.sh").is_file()),None)
assert ROOT is not None, "Run from the SC Linux checkout (make book-lab)"
def run(*argv):
    print("+"," ".join(map(str,argv)),flush=True)
    return subprocess.run([str(a) for a in argv],cwd=ROOT,check=True)
print("Source:",ROOT)

## Action 1 · Check that source and output are safe to use

In [ ]:
status=subprocess.run(["git","status","--porcelain","--untracked-files=normal"],cwd=ROOT,capture_output=True,text=True,check=True).stdout
assert not status.strip(), f"Commit source changes before build:\n{status}"
OUTPUT=ROOT/"build/output"
assert not OUTPUT.exists() or not any(OUTPUT.iterdir()), "Choose a fresh output directory; current build/output is not empty"
assert (ROOT/"build/builder.id").is_file(), "Run chapter 4 first"
print("Image destination:",OUTPUT)

## Action 2 · Validate trust before the privileged build

This validates certificate structure and its checked-in digest. It cannot establish your independent verification.

## Action 3 · Read the image-builder phases

`build-inside.sh` has named stages in execution order. The server RAW is finalized before its payload digest is embedded in the installer. Keeping this order prevents assembling an ISO with stale payload metadata.

In [ ]:
STAGES = ("verify_toolchain", "prepare_workspace", "validate_and_stage_inputs",
          "configure_efi_signing", "build_server_raw", "compress_server_raw",
          "record_installer_payload", "build_installer_uki",
          "assemble_installer_iso", "record_build_evidence")
pipeline = (ROOT/"scripts/build-inside.sh").read_text()
for position, stage in enumerate(STAGES, 1):
    assert f"{stage}()" in pipeline, f"Missing build stage: {stage}"
    print(f"{position:2}. {stage}")

## Action 4 · Inspect staged server files (optional)

This calls the **canonical preparation notebook** using the current Git revision. It writes only into ignored `build/` paths. The real build later stages its own fresh copy of committed source in the Docker container. This inspection is optional.

In [ ]:
import os
PREVIEW_STAGING = False
if PREVIEW_STAGING:
    source_sha = subprocess.check_output(["git","rev-parse","HEAD"],cwd=ROOT,text=True).strip()
    source_epoch = subprocess.check_output(["git","show","-s","--format=%ct","HEAD"],cwd=ROOT,text=True).strip()
    stage_env = dict(os.environ, SOURCE_COMMIT=source_sha, SOURCE_DATE_EPOCH=source_epoch)
    subprocess.run(["bash","scripts/notebook.sh","notebooks/prepare.ipynb"],
                   cwd=ROOT,env=stage_env,check=True)
    for image in ("server","installer"):
        overlay = ROOT/"build"/f"{image}-overlay"
        assert overlay.is_dir(), f"Staging failed: {overlay}"
        print(image, "staged files:", sum(p.is_file() for p in overlay.rglob("*")))
else:
    print("Optional overlay inspection skipped")

## Action 5 · Build RAW and ISO in one reviewed transaction

`make images` runs the checked stages inside a privileged Docker container. The output directory must be empty; image creation can use tens of GiB of temporary space.

In [ ]:
run("make","images")

## Action 6 · Confirm the server disk and its compressed payload

The uncompressed RAW is the whole GPT disk. The `.zst` file is the distributable compressed form of that same disk.

In [ ]:
raw=OUTPUT/"raw/sc-linux.raw"
compressed=OUTPUT/"sc-linux.raw.zst"
for name,artifact in (("Server RAW",raw),("Compressed RAW",compressed)):
    assert artifact.is_file() and artifact.stat().st_size > 1_048_576, f"Missing {name}"
    print(f"{name}: {artifact.stat().st_size:,} bytes")

## Action 7 · Inspect embedded payload metadata

The installer receives the size and SHA-256 of both RAW forms *before* its UKI is assembled. Chapter 6 independently re-hashes the artifacts.

In [ ]:
import json
payload=OUTPUT/"payload.json"
assert payload.is_file()
record=json.loads(payload.read_text())
assert set(record)=={"raw","compressed"}
assert record["raw"]["size"]==raw.stat().st_size
assert record["compressed"]["size"]==compressed.stat().st_size
for name,item in record.items():
    print(name, "bytes:", item["size"], "SHA-256:", item["sha256"])

## Action 8 · Check the installer and recorded provenance

Do not treat mere existence as boot verification. The QEMU acceptance test is a separate chapter.

In [ ]:
uki=OUTPUT/"installer/sc-linux-installer.efi"
iso=OUTPUT/"sc-linux.iso"
for label,path in (("Installer UKI",uki),("Installer ISO",iso)):
    assert path.is_file() and path.stat().st_size > 1_048_576, f"Missing {label}"
    print(label, f"{path.stat().st_size:,} bytes")
commit_file=OUTPUT/"source-commit.txt"
assert commit_file.is_file()
print("Source revision:", commit_file.read_text().strip())

## Check

The ordered stages produced a server RAW, its compressed payload, matching installer manifest, installer UKI and ISO. This is **image construction evidence**, not proof of a successful boot.

## Next

[Inspect the artifact checksums and footprint](06-artifacts.ipynb).